# Test: FLOWBOOK_UNCOPYABLE_AS_WRITE Behavior

This notebook demonstrates the difference between old and new handling of uncopyable variables.

## Expected Behavior

**Old behavior (default):** Uncopyable variables are removed from namespace.
- Cell B can be re-run without violation because Cell A never actually read `sock`.

**New behavior (`FLOWBOOK_UNCOPYABLE_AS_WRITE=1`):** Uncopyable variables are added to W (writes).
- Cell B re-run triggers a backward violation because Cell A read `sock` (which Cell B writes).

## Instructions

1. Run cells in order: B, A, then re-run B
2. Under old behavior: B re-runs successfully
3. Under new behavior: B re-run is REJECTED with backward violation

In [ ]:
# Cell A (position 0)
# This cell conditionally reads the socket if it exists.
# Under OLD behavior: socket was removed, so this branch is not taken.
# Under NEW behavior: socket exists, so we read it (establishing a dependency).

if 'sock' in dir():
    # Reading the socket - creates dependency
    sock_fd = sock.fileno()
    print(f"Socket exists with fd={sock_fd}")
else:
    sock_fd = -1
    print("Socket does not exist")

In [ ]:
# Cell B (position 1)
# Creates a socket - an uncopyable object.
#
# Under OLD behavior: sock is removed from namespace after checkpoint.
# Under NEW behavior: sock stays in namespace AND is added to W (writes).
#
# When re-running this cell after Cell A has read sock:
# - OLD: No conflict (A never read sock because it was removed)
# - NEW: Backward violation! (A read sock, B writes sock, A is earlier)

import socket
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
print(f"Created socket with fd={sock.fileno()}")

## How to Test

### Test Old Behavior (default)
```bash
jupyter lab
```
1. Run Cell B (creates socket, socket is removed)
2. Run Cell A (socket doesn't exist, prints "Socket does not exist")
3. Re-run Cell B → **Succeeds** (no violation)

### Test New Behavior
```bash
FLOWBOOK_UNCOPYABLE_AS_WRITE=1 jupyter lab
```
1. Run Cell B (creates socket, socket stays in namespace)
2. Run Cell A (socket exists, reads it, prints fd)
3. Re-run Cell B → **REJECTED** with backward violation

The violation message will indicate that Cell B writes `sock` which was read by Cell A.